# Akıllı Tarım Asistanı: STT ve NLP Deneyleri

Bu notebook Google Colab'da çalıştırıldı. Ses kayıtları (`akilli_tarim_test.wav`, `akilli_tarim_test_2.wav`) repoya eklenmedi.

1. Konuşmadan metne (STT) model karşılaştırması
2. Ek STT modelleri (Wav2Vec2, Canary)
3. Doğrusal SVM eğitimi ve ağırlıkların Java'ya aktarılması
4. YSA mimarisi

## 1. STT model karşılaştırması (60 komut)

Google on-device STT çıktısı ve Whisper modelleri, aynı Android ön işleme filtresinden geçirilip WER ile karşılaştırılır.

In [ ]:
# ==============================================================================
# 1. GEREKLİ KÜTÜPHANELERİN KURULUMU
# ==============================================================================
!pip install openai-whisper jiwer transformers torch librosa soundfile

import os
import torch
import librosa
import whisper
from jiwer import wer
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor

# Cihaz kontrolü: GPU (cuda) mevcutsa seçilir, yoksa CPU'da kalır
cihaz = "cuda" if torch.cuda.is_available() else "cpu"
print(f">>> ÇALIŞMA ORTAMI: {cihaz.upper()}")

# ==============================================================================
# 2. REFERANS METİNLERİN (GROUND TRUTH) BİRLEŞTİRİLMESİ
# ==============================================================================
metin_1 = "sistemi kontrol et durum ne guncel durum nedir sicaklik kac derece hava kac derece dereceyi soyler misin nem nasil toprak kuru mu nem oranini oku otonom moda gec akilli modu ac otomatik sulamayi baslat turp tarifine gec ev bitkisi sec tarifi degistirir misin pompayi ac suyu aciver motoru calistirir misin ciceklere su ver pompayi iki dakika boyunca calistir bitkileri sula suyu kapat pompayi durdur sulamayi kes motoru kapatir misin manuel moda gec el ile kontrole gec otonomdan cikar misin sistemi bana birak"
metin_2 = "motoru devreye al sulamaya basla lutfen pompayi calistir hadi suyu ac vanayi ac su pompasini ac sulamayi baslatsana biraz su ver sistemi sula sulama islemini bitir su pompasini durdur sulamayi durdursana artik su verme sistemin suyunu kes pompayi sustur duraklat bitki nasil sensor oku ne alemdeyiz rapor ver bitkilerin durumu ne sera ne alemde durum raporu alayim icerisi sicak mi isi durumu nasil kac derece oldu ortam isisi ne suan hava kac derece icerisi soguk mu"

gercek_metin_toplam = metin_1 + " " + metin_2

# ==============================================================================
# 3. ANDROID UYUMLU ÖN İŞLEME (PREPROCESS) FONKSİYONU
# ==============================================================================
def nlp_preprocess(text):
    if text is None:
        return ""
    processed = text.lower() \
                    .replace("ı", "i") \
                    .replace("ğ", "g") \
                    .replace("ü", "u") \
                    .replace("ş", "s") \
                    .replace("ö", "o") \
                    .replace("ç", "c")

    noktalama_ve_semboller = ["?", ",", ".", "!", "-", ";", ":"]
    for sembol in noktalama_ve_semboller:
        processed = processed.replace(sembol, "")

    return " ".join(processed.split()).strip()

gercek_temiz = nlp_preprocess(gercek_metin_toplam)

# ==============================================================================
# 4. GOOGLE STT ANDROID CİHAZ İÇİ HAM LOG VERİLERİ
# ==============================================================================
google_ham_1 = "sistemi kontrol et durum ne? güncel durum nedir? sıcaklık kaç derece? hava kaç derece? dereceği söyler misin? nem nasıl? toprak kurumu? nem oranını oku? otonom moda geç? akıllı modu aç? otomatik sulamayı başlat? turb tarifine geç? ev bitkisini seç? tarifi değiştirir misin? pompayı aç? suyu açıver? modoru çalıştırır mısın? çiçeklere su ver? pompayı iki dakika boyunca çalıştır? bitkileri sula? suyu kapat? pompayı durdur? sula mayi kes? modoru kapatır mısın? monal moda geç? monal moda geç? el ile kontrol e geç? otonomdan çıkarmışsın? sistemi bana bırak"
google_ham_2 = "motoru devreye al sulamaya başla lütfen pompayı çalıştır hadi suyu aç vanayı aç su pompasını aç sulamayı başlatsana biraz su ver sistemi sula sulama işlemini bitir su pompasını durdur sulamayı durdursana artık su verme sistemin suyunu kes pompayı sustur duraklat bitki nasıl sensör oku ne alemdeyiz rapor ver bitkilerin durumu ne sera ne alemde durum raporu alayım içerisi sıcak mı ısı durumu nasıl kaç derece oldu ortam ısısı ne şuan hava kaç derece içerisi soğuk mu"

google_stt_toplam_ham = google_ham_1 + " " + google_ham_2

# ==============================================================================
# 5. SES DOSYALARI VE MODELLERİN ÇALIŞTIRILMASI
# ==============================================================================
ses_dosyalari = ["akilli_tarim_test.wav", "akilli_tarim_test_2.wav"]

# --- WHISPER TRANSCRIPTION FONKSİYONU (GPU & FP16 Destekli) ---
def transcribe_whisper(model_size, ses_listesi):
    try:
        print(f"--> Whisper '{model_size}' modeli yükleniyor ve sesler işleniyor...")
        model = whisper.load_model(model_size, device=cihaz)
        metin_sonuc = ""
        for ses in ses_listesi:
            if os.path.exists(ses):
                result = model.transcribe(ses, language="tr", fp16=(cihaz == "cuda"))
                metin_sonuc += result["text"] + " "
            else:
                print(f"-> Uyarı: {ses} dosyası bulunamadı!")
        return metin_sonuc.strip()
    except Exception as e:
        return f"Whisper {model_size} hatası: {e}"

# --- META WAV2VEC2 TRANSCRIPTION FONKSİYONU ---
def transcribe_wav2vec2(ses_listesi):
    try:
        print("--> Meta Wav2Vec2 (MMS-1B) modeli yükleniyor ve sesler işleniyor...")
        processor = Wav2Vec2Processor.from_pretrained("facebook/mms-1b-all")
        model_w2v = Wav2Vec2ForCTC.from_pretrained("facebook/mms-1b-all").to(cihaz)
        processor.tokenizer.set_target_lang("tur")
        model_w2v.load_adapter("tur")

        w2v_sonuc = ""
        for ses in ses_listesi:
            if os.path.exists(ses):
                # Wav2Vec2 16kHz ses frekansı bekler, librosa ile downsample yapıyoruz
                ses_sinyali, _ = librosa.load(ses, sr=16000)
                girdi_degerleri = processor(ses_sinyali, sampling_rate=16000, return_tensors="pt").input_values.to(cihaz)

                with torch.no_grad():
                    logits = model_w2v(girdi_degerleri).logits

                tahmin_idleri = torch.argmax(logits, dim=-1)
                transkripsiyon = processor.batch_decode(tahmin_idleri)[0]
                w2v_sonuc += transkripsiyon + " "
        return w2v_sonuc.strip()
    except Exception as e:
        return f"Meta Wav2Vec2 hatası: {e}"

# Modelleri sırayla koşturuyoruz
print(">>> 1/4: Google STT verileri hazırlanıyor...")
# Google STT doğrudan ham veri bloğundan okunacak

print(">>> 2/4: Whisper 'tiny' ve 'base' modelleri çalıştırılıyor...")
whisper_tiny_cikti = transcribe_whisper("tiny", ses_dosyalari)
whisper_base_cikti = transcribe_whisper("base", ses_dosyalari)

print(">>> 3/4: Whisper 'small' ve 'medium' modelleri çalıştırılıyor...")
whisper_small_cikti = transcribe_whisper("small", ses_dosyalari)
whisper_medium_cikti = transcribe_whisper("medium", ses_dosyalari)

print(">>> 4/4: Meta Wav2Vec2 (MMS-1B-ALL) modeli çalıştırılıyor...")
meta_w2v_cikti = transcribe_wav2vec2(ses_dosyalari)

# ==============================================================================
# 6. KARŞILAŞTIRMALI BENCHMARK TABLOSU HESAPLAMA
# ==============================================================================
stt_modelleri = {
    "Google STT (On-Device)": google_stt_toplam_ham,
    "OpenAI Whisper (Tiny)": whisper_tiny_cikti,
    "OpenAI Whisper (Base)": whisper_base_cikti,
    "OpenAI Whisper (Small)": whisper_small_cikti,
    "OpenAI Whisper (Medium)": whisper_medium_cikti,
    "Meta Wav2Vec2 (MMS-1B)": meta_w2v_cikti
}

rapor_listesi = []
for ad, ham in stt_modelleri.items():
    temiz = nlp_preprocess(ham)
    hata = wer(gercek_temiz, temiz)
    rapor_listesi.append({
        "model": ad,
        "wer": max(0.0, hata * 100),
        "accuracy": max(0.0, (1 - hata) * 100)
    })

# ==============================================================================
# 7. RAPOR ÇIKTISI
# ==============================================================================
print("\n" + "="*80)
print("     60 KOMUTLUK GENİŞLETİLMİŞ AKILLI TARIM STT ENTEGRASYON BENCHMARK'I")
print("="*80)
print(f"{'STT Model Mimarisi':<25} | {'Word Error Rate (WER)':<22} | {'Doğruluk Oranı (Accuracy)':<25}")
print("-"*80)
for r in rapor_listesi:
    print(f"{r['model']:<25} | %{r['wer']:<20.2f} | %{r['accuracy']:<23.2f}")
print("="*80)
print("Not: Tüm modellerin ham çıktıları Android Preprocess filtresinden geçirilmiştir.")

>>> ÇALIŞMA ORTAMI: CUDA
>>> 1/4: Google STT verileri hazırlanıyor...
>>> 2/4: Whisper 'tiny' ve 'base' modelleri çalıştırılıyor...
--> Whisper 'tiny' modeli yükleniyor ve sesler işleniyor...
--> Whisper 'base' modeli yükleniyor ve sesler işleniyor...
>>> 3/4: Whisper 'small' ve 'medium' modelleri çalıştırılıyor...
--> Whisper 'small' modeli yükleniyor ve sesler işleniyor...
--> Whisper 'medium' modeli yükleniyor ve sesler işleniyor...
>>> 4/4: Meta Wav2Vec2 (MMS-1B-ALL) modeli çalıştırılıyor...
--> Meta Wav2Vec2 (MMS-1B) modeli yükleniyor ve sesler işleniyor...



     60 KOMUTLUK GENİŞLETİLMİŞ AKILLI TARIM STT ENTEGRASYON BENCHMARK'I
STT Model Mimarisi        | Word Error Rate (WER)  | Doğruluk Oranı (Accuracy)
--------------------------------------------------------------------------------
Google STT (On-Device)    | %10.76                | %89.24                  
OpenAI Whisper (Tiny)     | %31.01                | %68.99                  
OpenAI Whisper (Base)     | %16.46                | %83.54                  
OpenAI Whisper (Small)    | %10.13                | %89.87                  
OpenAI Whisper (Medium)   | %5.06                 | %94.94                  
Meta Wav2Vec2 (MMS-1B)    | %8.86                 | %91.14                  
Not: Tüm modellerin ham çıktıları Android Preprocess filtresinden geçirilmiştir.


## 2. Ek STT modelleri

In [ ]:
# ==============================================================================
# 1. GEREKLİ KÜTÜPHANELERİN KURULUMU
# ==============================================================================
!pip install transformers torch jiwer librosa soundfile

import os
import torch
import librosa
from jiwer import wer
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor, pipeline

# ==============================================================================
# 2. REFERANS METİNLERİN (GROUND TRUTH) TANIMLANMASI
# ==============================================================================
metin_1 = "sistemi kontrol et durum ne guncel durum nedir sicaklik kac derece hava kac derece dereceyi soyler misin nem nasil toprak kuru mu nem oranini oku otonom moda gec akilli modu ac otomatik sulamayi baslat turp tarifine gec ev bitkisi sec tarifi degistirir misin pompayi ac suyu aciver motoru calistirir misin ciceklere su ver pompayi iki dakika boyunca calistir bitkileri sula suyu kapat pompayi durdur sulamayi kes motoru kapatir misin manuel moda gec el ile kontrole gec otonomdan cikar misin sistemi bana birak"
metin_2 = "motoru devreye al sulamaya basla lutfen pompayi calistir hadi suyu ac vanayi ac su pompasini ac sulamayi baslatsana biraz su ver sistemi sula sulama islemini bitir su pompasini durdur sulamayi durdursana artik su verme sistemin suyunu kes pompayi sustur duraklat bitki nasil sensor oku ne alemdeyiz rapor ver bitkilerin durumu ne sera ne alemde durum raporu alayim icerisi sicak mi isi durumu nasil kac derece oldu ortam isisi ne suan hava kac derece icerisi soguk mu"

gercek_metin_toplam = metin_1 + " " + metin_2

# Android Ön İşleme Filtresi
def nlp_preprocess(text):
    if text is None: return ""
    processed = text.lower() \
                    .replace("ı", "i") \
                    .replace("ğ", "g") \
                    .replace("ü", "u") \
                    .replace("ş", "s") \
                    .replace("ö", "o") \
                    .replace("ç", "c")
    noktalama_ve_semboller = ["?", ",", ".", "!", "-", ";", ":"]
    for sembol in noktalama_ve_semboller:
        processed = processed.replace(sembol, "")
    return " ".join(processed.split()).strip()

gercek_temiz = nlp_preprocess(gercek_metin_toplam)
ses_dosyalari = ["akilli_tarim_test.wav", "akilli_tarim_test_2.wav"]
cihaz = "cuda" if torch.cuda.is_available() else "cpu"

# ==============================================================================
# 3. ALTERNATİF MODEL 1: META WAV2VEC2 TR (MMS-1B-ALL) TESTİ
# ==============================================================================
print(">>> 1/2: Meta Wav2Vec2 Türkçe modeli yükleniyor ve sesler işleniyor...")
try:
    processor = Wav2Vec2Processor.from_pretrained("facebook/mms-1b-all")
    model_w2v = Wav2Vec2ForCTC.from_pretrained("facebook/mms-1b-all").to(cihaz)
    processor.tokenizer.set_target_lang("tur")
    model_w2v.load_adapter("tur")

    w2v_cikti = ""
    for ses in ses_dosyalari:
        if os.path.exists(ses):
            # Wav2Vec2 kesinlikle 16kHz ses sinyali bekler, librosa ile yeniden örnekliyoruz
            ses_sinyali, _ = librosa.load(ses, sr=16000)
            girdi_degerleri = processor(ses_sinyali, sampling_rate=16000, return_tensors="pt").input_values.to(cihaz)

            with torch.no_grad():
                logits = model_w2v(girdi_degerleri).logits

            tahmin_idleri = torch.argmax(logits, dim=-1)
            transkripsiyon = processor.batch_decode(tahmin_idleri)[0]
            w2v_cikti += transkripsiyon + " "
    w2v_cikti = w2v_cikti.strip()
except Exception as e:
    w2v_cikti = ""
    print(f"Meta Wav2Vec2 hatası: {e}")

# ==============================================================================
# 4. ALTERNATİF MODEL 2: NVIDIA CANARY (1B) TESTİ
# ==============================================================================
print(">>> 2/2: NVIDIA Canary-1B Hugging Face hattı kuruluyor...")
try:
    # NVIDIA'nın en optimize transkripsiyon boru hattını çağırıyoruz
    canary_pipeline = pipeline("automatic-speech-recognition", model="nvidia/canary-1b", device=cihaz)

    canary_cikti = ""
    for ses in ses_dosyalari:
        if os.path.exists(ses):
            # Canary modeli büyük olduğu için chunk_length_s belirleyerek hafıza yönetimini sağlıyoruz
            result = canary_pipeline(ses, generate_kwargs={"task": "transcribe", "language": "tr"})
            canary_cikti += result["text"] + " "
    canary_cikti = canary_cikti.strip()
except Exception as e:
    canary_cikti = ""
    print(f"NVIDIA Canary hatası: {e}")

# ==============================================================================
# 5. METRİK HESAPLAMA VE EK DOSYA EKİ TABLOSU
# ==============================================================================
ek_modeller = {
    "Meta Wav2Vec2 (MMS-1B)": w2v_cikti,
    "NVIDIA Canary (1B)": canary_cikti
}

print("\n" + "="*80)
print("             EK ALTERNATİF STT MODELLERİ BENCHMARK SONUÇLARI")
print("===============================================================================")
print(f"{'STT Model Mimarisi':<25} | {'Word Error Rate (WER)':<22} | {'Doğruluk Oranı (Accuracy)':<25}")
print("-"*80)

for ad, ham in ek_modeller.items():
    if ham:
        temiz = nlp_preprocess(ham)
        hata = wer(gercek_temiz, temiz)
        print(f"{ad:<25} | %{hata * 100:<20.2f} | %{(1 - hata) * 100:<23.2f}")
    else:
        print(f"{ad:<25} | {'Hata Oluştu/Çalışmadı':<22} | {'-':<25}")
print("="*80)

>>> 1/2: Meta Wav2Vec2 Türkçe modeli yükleniyor ve sesler işleniyor...


>>> 2/2: NVIDIA Canary-1B Hugging Face hattı kuruluyor...
NVIDIA Canary hatası: Unrecognized model in nvidia/canary-1b. Should have a `model_type` key in its config.json.

             EK ALTERNATİF STT MODELLERİ BENCHMARK SONUÇLARI
STT Model Mimarisi        | Word Error Rate (WER)  | Doğruluk Oranı (Accuracy)
--------------------------------------------------------------------------------
Meta Wav2Vec2 (MMS-1B)    | %8.86                 | %91.14                  
NVIDIA Canary (1B)        | Hata Oluştu/Çalışmadı  | -                        


## 3. Doğrusal SVM: eğitim ve Java'ya aktarım

Karakter n-gram TF-IDF + `LinearSVC`. Çıktıdaki ağırlıklar Android tarafındaki `CommandProcessor` sınıfına kopyalanır. (Bu, sonradan geri alınan optimizasyon denemesidir.)

In [ ]:
import numpy as np
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Genişletilmiş Veri Kümesi
dataset = [
    ("pompayi ac", 0), ("suyu ac", 0), ("sulamayi baslat", 0), ("pompayi acar misin", 0), ("suyu acar misin", 0),
    ("suyu kapat", 1), ("pompayi durdur", 1), ("sulamayi kes", 1), ("suyu kapatir misin", 1),
    ("durum ne", 2), ("sistem nasil", 2), ("degerleri oku", 2), ("durumu soyler misin", 2),
    ("sicaklik kac", 3), ("derece ne", 3), ("hava kac derece", 3), ("dereceyi soyler misin", 3),
    ("nem nasil", 4), ("toprak kuru mu", 4), ("nem degeri", 4), ("nemi soyler misin", 4),
    ("otonom moda gec", 5), ("akilli modu ac", 5), ("otonom moda gecer misin", 5),
    ("manuel moda gec", 6), ("el ile kontrole gec", 6), ("manuel moda gecer misin", 6),
    ("turp tarifine gec", 7), ("ev bitkisi sec", 7), ("tarif degistir", 7), ("turp tarifine gecer misin", 7)
]
sentences, labels = zip(*dataset)

# 2. Harf Bazlı Gelişmiş Vektörleştirici (Char N-Gram)
# Kelimelerin eklerini (mısın, açar vb.) harf köklerinden yakalamak için 3'lü ve 4'lü harf bloklarına ayırır
vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(3, 4))
X = vectorizer.fit_transform(sentences).toarray()
y = np.array(labels)

# 3. Esnek Marjinli SVM Eğitimi (C=0.5 hata toleransını artırır)
svm = LinearSVC(C=0.5, loss='squared_hinge', dual=False, max_iter=10000)
svm.fit(X, y)

# ==============================================================================
# 4. YENİ ANDROID PARAMETRE GENERATOR
# ==============================================================================
print("// --- YENİ MATRİS BOYUTLARI ---")
print(f"NUM_CLASSES = {svm.coef_.shape[0]};")
print(f"FEATURE_SIZE = {svm.coef_.shape[1]};\n")

print("private static void initializeSvmParameters() {")
for i, b in enumerate(svm.intercept_):
    print(f"    SVM_BIASES[{i}] = {b:.6f}f;")

print("\n")
for c in range(svm.coef_.shape[0]):
    for f in range(svm.coef_.shape[1]):
        weight_val = svm.coef_[c][f]
        if abs(weight_val) > 1e-4:
            print(f"    SVM_WEIGHTS[{c}][{f}] = {weight_val:.6f}f;")
print("}")

print("\n// --- JAVA İÇİN SÖZLÜK (N-GRAM ŞABLONU) ---")
for char_seq, idx in vectorizer.vocabulary_.items():
    # Java tarafında kullanılacak karakter dizin haritası
    pass

// --- YENİ MATRİS BOYUTLARI ---
NUM_CLASSES = 8;
FEATURE_SIZE = 392;

private static void initializeSvmParameters() {
    SVM_BIASES[0] = -0.672142f;
    SVM_BIASES[1] = -0.680716f;
    SVM_BIASES[2] = -0.645798f;
    SVM_BIASES[3] = -0.652955f;
    SVM_BIASES[4] = -0.645673f;
    SVM_BIASES[5] = -0.751120f;
    SVM_BIASES[6] = -0.748604f;
    SVM_BIASES[7] = -0.652915f;


    SVM_WEIGHTS[0][0] = 0.536344f;
    SVM_WEIGHTS[0][1] = 0.252735f;
    SVM_WEIGHTS[0][2] = 0.197374f;
    SVM_WEIGHTS[0][3] = 0.197374f;
    SVM_WEIGHTS[0][4] = -0.034676f;
    SVM_WEIGHTS[0][5] = -0.034676f;
    SVM_WEIGHTS[0][6] = -0.069966f;
    SVM_WEIGHTS[0][7] = -0.055336f;
    SVM_WEIGHTS[0][8] = -0.023715f;
    SVM_WEIGHTS[0][9] = -0.098932f;
    SVM_WEIGHTS[0][10] = -0.098932f;
    SVM_WEIGHTS[0][11] = -0.088293f;
    SVM_WEIGHTS[0][12] = -0.088293f;
    SVM_WEIGHTS[0][13] = -0.025567f;
    SVM_WEIGHTS[0][14] = -0.025567f;
    SVM_WEIGHTS[0][15] = -0.149642f;
    SVM_WEIGHTS[0][16] = -0.051769f;
    SVM_

## 4. YSA mimarisi

Uygulamada kullanılan TFLite modelinin mimarisi ve parametre özeti.

In [ ]:
# ==============================================================================
# GÜNCELLENMİŞ YSA MODELİ GÖRSELLEŞTİRME VE ÖZET BETİĞİ (Keras 3 UYUMLU)
# ==============================================================================
import tensorflow as tf
from tensorflow.keras.utils import plot_model

# 1. Parametre Tanımlamaları
word_count = 221  # Sözlükteki toplam kelime sayısı
max_len = 5       # Cümle uzunluğu (MAX_LEN)
num_classes = 8   # 8 Farklı Intent Sınıfı

# 2. Model Mimarisi
ysa_model = tf.keras.Sequential([
    # Keras 3+ standartlarına uygun Input katmanını açıkça tanımlıyoruz
    tf.keras.layers.Input(shape=(max_len,), dtype='int32'),

    # Kelime ID'lerini anlamsal 16 boyutlu vektörlere dönüştüren katman
    tf.keras.layers.Embedding(input_dim=word_count, output_dim=16),

    # Vektör ortalamasını alan katman
    tf.keras.layers.GlobalAveragePooling1D(),

    # Gizli katman
    tf.keras.layers.Dense(16, activation='relu'),

    # Çıkış katmanı
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

# CRITICAL: Yeni Keras sürümleri için modeli açıkça inşa (build) ediyoruz
ysa_model.build(input_shape=(None, max_len))

ysa_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# ==============================================================================
# [ÇÖZÜM 3] - MODEL SUMMARY: METİN TABANLI HIZLI ÖZET
# ==============================================================================
print("\n" + "="*60)
print("             YAPAY SİNİR AĞI KATMAN VE PARAMETRE ÖZETİ")
print("========================================================")
ysa_model.summary()
print("========================================================\n")

# ==============================================================================
# [ÇÖZÜM 2] - KERAS PLOT_MODEL: GÖRSEL AKIŞ ŞEMASI ÜRETİMİ
# ==============================================================================
print(">>> Akıllı Tarım NLP modeli görselleştiriliyor...")

# Model şemasını 'akilli_tarim_ysa_mimarisi.png' adıyla kaydeder
plot_model(
    ysa_model,
    to_file='akilli_tarim_ysa_mimarisi.png',
    show_shapes=True,
    show_layer_names=True,
    show_layer_activations=True # Katmanların aktivasyon fonksiyonlarını da şemaya ekler
)

print(">>> Başarılı! Görsel 'akilli_tarim_ysa_mimarisi.png' adıyla sol paneldeki dosya alanına kaydedildi.")


             YAPAY SİNİR AĞI KATMAN VE PARAMETRE ÖZETİ


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 5, 16)          │         3,536 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_1      │ (None, 16)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 8)              │           136 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,944 (15.41 KB)

 Trainable params: 3,944 (15.41 KB)

 Non-trainable params: 0 (0.00 B)


>>> Akıllı Tarım NLP modeli görselleştiriliyor...
>>> Başarılı! Görsel 'akilli_tarim_ysa_mimarisi.png' adıyla sol paneldeki dosya alanına kaydedildi.
